# 1) LOAD TEST SET

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, optimizers
from autoencoder import Autoencoder


IMG_HEIGHT = 224
IMG_WIDTH  = 224
AUTOTUNE   = tf.data.AUTOTUNE
DATASET_BASE = '../../data'         
OBJ           = 'leather'
MODEL_PATH    = '/Users/vegardhaugerud/mvtec_anomaly_detection_project/notebooks/Autoencoder/checkpoints/autoencoder_01_val5755.8306.keras'        
OUT_BASE      = '../../output/anomaly_maps'                  
IMG_SIZE      = (224, 224)

model = Autoencoder()

model.load_weights(MODEL_PATH)


/Users/vegardhaugerud/.pyenv/versions/3.11.4/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:99: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(


In [2]:
raw_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    '../../data/leather/test',
    labels='inferred',
    label_mode='int',            # 0 or 1 per subfolder
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=1,
    shuffle=False
)
class_names = raw_test_ds.class_names
print("class_names:", class_names)  

normalize = tf.keras.layers.Rescaling(1./255)
test_ds = (
    raw_test_ds
     .map(lambda x,y: (normalize(x), y), num_parallel_calls=AUTOTUNE)
)

Found 124 files belonging to 6 classes.
class_names: ['color', 'cut', 'fold', 'glue', 'good', 'poke']


# 2) NORMALIZE


In [3]:
normalize = tf.keras.layers.Rescaling(1.0/255)

test_ds = test_ds.map(lambda x, y: (normalize(x), y),num_parallel_calls=AUTOTUNE)


# 3) INFERENCE & ERROR COMPUTATION


In [4]:
import numpy as np
from tqdm import tqdm 

errors = []
labels = []

for x_batch, y_batch in tqdm(test_ds, total=len(raw_test_ds)):
    x_rec = model(x_batch, training=False)
    mse = tf.reduce_mean(tf.square(x_batch - x_rec), axis=[1,2,3])
    errors.append(mse.numpy()[0])
    labels.append(int(y_batch.numpy()[0]))

errors = np.array(errors)
labels = np.array(labels)

print(f"Test set size: {len(errors)} images")


100%|██████████| 124/124 [00:28<00:00,  4.36it/s]

Test set size: 124 images


# 4) Creating anomaly maps



In [5]:
# gen_maps.py
import os
import numpy as np
from PIL import Image
import tifffile
import tensorflow as tf


test_root = os.path.join(DATASET_BASE, OBJ, 'test')
for defect in os.listdir(test_root):            # defect = 'good' or 'hole'
    src_dir = os.path.join(test_root, defect)
    dst_dir = os.path.join(OUT_BASE, OBJ, 'test', defect)
    os.makedirs(dst_dir, exist_ok=True)

    for fname in os.listdir(src_dir):
        if not fname.lower().endswith('.png'):
            continue

        # load & preprocess
        img = Image.open(os.path.join(src_dir, fname)).convert('RGB')
        img = img.resize(IMG_SIZE)
        arr = np.array(img, dtype=np.float32)   # H×W×3, [0–255]
        arr = arr[None, ...]  # add batch dimension
        arr = normalize(arr)                    # → [0–1]

        # reconstruct & compute per-pixel MSE
        rec = model(arr, training=False).numpy()[0]  
        amap = np.mean((arr[0] - rec)**2, axis=-1)  # H×W

        # save as float32 TIFF
        out_path = os.path.join(dst_dir, fname.replace('.png','') + '.tiff')
        tifffile.imwrite(out_path, amap.astype(np.float32))

print("Done generating anomaly maps.")


Done generating anomaly maps.


# 5) Running eval function


In [ ]:
import subprocess

def run_leather_evaluation():
    cmd = [
        "python", "../../evaluate/patchCore_Eval/evaluate_experiment2.py",
        "--dataset_base_dir", "../../data",
        "--anomaly_maps_dir", "../../output/anomaly_maps",
        "--output_dir", "../../output/metrics",
        "--evaluated_objects", "leather",
        "--pro_integration_limit", "0.3"
    ]
    subprocess.run(cmd, check=True)

if __name__ == "__main__":
    run_leather_evaluation()
    print("Evaluation completed successfully.")

python: can't open file '/Users/vegardhaugerud/mvtec_anomaly_detection_project/notebooks/Autoencoder/../evaluate/patchCore_Eval/evaluate_experiment2.py': [Errno 2] No such file or directory


CalledProcessError: Command '['python', '../evaluate/patchCore_Eval/evaluate_experiment2.py', '--dataset_base_dir', '../data', '--anomaly_maps_dir', '../output/anomaly_maps', '--output_dir', '../output/metrics', '--evaluated_objects', 'leather', '--pro_integration_limit', '0.3']' returned non-zero exit status 2.